# S01 · Suma de vectores en CUDA (Colab / Kaggle)

> **Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1** · Unidad V · Sesión S01 · lunes 23 de noviembre de 2026 · 10:30–12:00 · Salón 3208

**Antes de correr nada, activa la GPU:**

- **Colab:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: GPU (T4)* → Guardar.
- **Kaggle:** panel derecho *Settings → Accelerator → GPU* (T4 x2 o P100). Kaggle pide verificar el teléfono para usar GPU.

La primera celda debe mostrar una tarjeta NVIDIA. Si dice `command not found` o `No devices were found`, no hay GPU: vuelve a cambiar el tipo de entorno. Si Colab avisa que no hay GPU disponible, trabaja la correctitud con `NUMBA_ENABLE_CUDASIM=1` en tu laptop (celdas marcadas) y reintenta más tarde; la cuota se libera en horas.

Sigue el `README.md` de la práctica guiada: **predice por escrito antes de ejecutar cada celda.**

In [ ]:
!nvidia-smi

In [ ]:
# Arquitectura para nvcc: T4 (Colab) -> sm_75; P100 (Kaggle) -> sm_60. Se detecta; si falla, queda sm_75.
import subprocess
ARCH = "sm_75"
try:
    cap = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                  text=True).strip().splitlines()[0]
    ARCH = "sm_" + cap.replace(".", "")
except Exception as exc:  # sin GPU o nvidia-smi viejo
    print("No se pudo detectar la GPU:", exc)
print("ARCH =", ARCH)

## 1. Escribir los fuentes en el disco de la sesión

Cada celda `%%writefile` guarda un archivo. Son los mismos archivos de `practica-guiada/`; si los editas aquí, vuelve a ejecutar la celda y recompila.

In [ ]:
%%writefile vector_add.cu
/*
 * vector_add.cu — Suma de vectores c = a + b en CUDA C (Unidad V, S01).
 *
 * Uso:  ./vector_add [n] [block] [grid] [archivo.csv]
 *   n        elementos por vector (por defecto 16 777 216 = 2^24, 64 MB por vector)
 *   block    hilos por bloque (por defecto 256)
 *   grid     bloques a lanzar; 0 = calcular (n + block - 1) / block.
 *            Otro valor fuerza el lanzamiento (sirve para experimentar en la fase Investigar).
 *   csv      si se da, agrega una línea con la medición (lo usa el notebook)
 *
 * Compilar y correr (Colab/Kaggle con GPU):
 *   nvcc -O2 -arch=sm_75 vector_add.cu -o vector_add && ./vector_add
 * Verificar que compila sin GPU (entorno sin tarjeta):
 *   nvcc -c -arch=sm_75 vector_add.cu -o vector_add.o
 * Variante deliberadamente incorrecta (fase Investigar: ¿qué pasa sin la guardia?):
 *   nvcc -O2 -arch=sm_75 -DSIN_GUARDIA vector_add.cu -o vector_add_sin_guardia
 *
 * Medición: protocolo del curso (calentamiento + 5 repeticiones, mediana) con cudaEvent_t.
 * Se reporta el tiempo del kernel solo y el tiempo con las copias anfitrión<->dispositivo.
 */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <cuda_runtime.h>

#define REPS 5

/* Toda llamada a la API de CUDA regresa un cudaError_t. Esta macro lo revisa y
 * detiene el programa con el archivo y la línea donde falló. */
#define CUDA_CHECK(call)                                                           \
    do {                                                                           \
        cudaError_t err__ = (call);                                                \
        if (err__ != cudaSuccess) {                                                \
            fprintf(stderr, "Error de CUDA en %s:%d: %s\n", __FILE__, __LINE__,   \
                    cudaGetErrorString(err__));                                    \
            exit(EXIT_FAILURE);                                                    \
        }                                                                          \
    } while (0)

/* ---------- Código de dispositivo (corre en la GPU) ---------- */

/* Cada hilo calcula UN elemento. El índice global se arma con la posición del
 * bloque en la malla (blockIdx.x), el tamaño del bloque (blockDim.x) y la
 * posición del hilo dentro del bloque (threadIdx.x). */
__global__ void vector_add(const float *a, const float *b, float *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
#ifndef SIN_GUARDIA
    if (i < n) /* guardia: los hilos sobrantes del último bloque no tocan memoria */
#endif
        c[i] = a[i] + b[i];
}

/* ---------- Código de anfitrión (corre en la CPU) ---------- */

/* Referencia secuencial: misma aritmética, mismo orden. Contra esto se verifica. */
static void vector_add_cpu(const float *a, const float *b, float *c, int n)
{
    for (int i = 0; i < n; i++)
        c[i] = a[i] + b[i];
}

static int cmp_float(const void *x, const void *y)
{
    float fx = *(const float *)x, fy = *(const float *)y;
    return (fx > fy) - (fx < fy);
}

/* Ordena en su lugar y regresa la mediana (REPS es impar). */
static float median_ms(float *v, int k)
{
    qsort(v, (size_t)k, sizeof(float), cmp_float);
    return v[k / 2];
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : (1 << 24);
    int block = (argc > 2) ? atoi(argv[2]) : 256;
    int grid = (argc > 3) ? atoi(argv[3]) : 0;
    const char *csv_path = (argc > 4) ? argv[4] : NULL;

    /* No se limita block a 1024 a propósito: con block=2000 el lanzamiento es inválido y
     * cudaGetLastError lo reporta ("invalid configuration argument"). Ver fase Investigar. */
    if (n <= 0 || block <= 0) {
        fprintf(stderr, "uso: %s [n>0] [block>0; máximo 1024 en la GPU] [grid; 0=auto] [archivo.csv]\n", argv[0]);
        return 1;
    }
    if (grid <= 0)
        grid = (n + block - 1) / block; /* división entera redondeada hacia arriba */

    size_t bytes = (size_t)n * sizeof(float);

    /* Memoria del anfitrión */
    float *h_a = (float *)malloc(bytes);
    float *h_b = (float *)malloc(bytes);
    float *h_c = (float *)malloc(bytes);
    float *h_ref = (float *)malloc(bytes);
    if (!h_a || !h_b || !h_c || !h_ref) {
        fprintf(stderr, "Sin memoria en el anfitrión para n=%d\n", n);
        return 1;
    }
    /* Valores enteros pequeños: la suma en float es exacta y el checksum es reproducible. */
    for (int i = 0; i < n; i++) {
        h_a[i] = (float)(i % 7);
        h_b[i] = (float)(i % 13);
    }
    memset(h_c, 0, bytes);

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("vector_add (CUDA C): n=%d block=%d grid=%d hilos_lanzados=%lld gpu=\"%s\" (SMs=%d, cc %d.%d)\n",
           n, block, grid, (long long)grid * block, prop.name, prop.multiProcessorCount,
           prop.major, prop.minor);

    /* Memoria del dispositivo */
    float *d_a = NULL, *d_b = NULL, *d_c = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemset(d_c, 0, bytes));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    /* ---- Medición 1: solo el kernel ---- */
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    vector_add<<<grid, block>>>(d_a, d_b, d_c, n); /* calentamiento (no se cuenta) */
    CUDA_CHECK(cudaGetLastError());      /* ¿falló el LANZAMIENTO? (configuración inválida) */
    CUDA_CHECK(cudaDeviceSynchronize()); /* ¿falló la EJECUCIÓN? (p. ej. acceso ilegal a memoria) */

    float t_kernel[REPS];
    for (int r = 0; r < REPS; r++) {
        CUDA_CHECK(cudaEventRecord(start));
        vector_add<<<grid, block>>>(d_a, d_b, d_c, n);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaEventElapsedTime(&t_kernel[r], start, stop)); /* milisegundos */
    }
    CUDA_CHECK(cudaGetLastError());

    /* ---- Medición 2: copias H2D + kernel + copia D2H ---- */
    float t_total[REPS];
    for (int r = 0; r < REPS; r++) {
        CUDA_CHECK(cudaEventRecord(start));
        CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));
        vector_add<<<grid, block>>>(d_a, d_b, d_c, n);
        CUDA_CHECK(cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaEventElapsedTime(&t_total[r], start, stop));
    }
    CUDA_CHECK(cudaGetLastError());

    /* ---- Verificación contra la CPU: "sin error" no significa "correcto" ---- */
    vector_add_cpu(h_a, h_b, h_ref, n);
    double max_abs_err = 0.0, checksum = 0.0;
    for (int i = 0; i < n; i++) {
        double diff = fabs((double)h_c[i] - (double)h_ref[i]);
        if (diff > max_abs_err)
            max_abs_err = diff;
        checksum += h_c[i];
    }
    int ok = (max_abs_err <= 1e-5);

    float k_med = median_ms(t_kernel, REPS), k_min = t_kernel[0], k_max = t_kernel[REPS - 1];
    float t_med = median_ms(t_total, REPS), t_min = t_total[0], t_max = t_total[REPS - 1];
    double gb_moved = 3.0 * (double)bytes / 1e9; /* lee a y b, escribe c */
    double bw = gb_moved / (k_med / 1e3);

    printf("kernel_mediana_ms=%.4f (min %.4f, max %.4f)  con_copias_mediana_ms=%.3f (min %.3f, max %.3f)  "
           "ancho_banda_efectivo_GB_s=%.1f\n",
           k_med, k_min, k_max, t_med, t_min, t_max, bw);
    printf("checksum=%.1f max_abs_err=%.3g verificacion=%s\n", checksum, max_abs_err, ok ? "OK" : "FALLA");

    if (csv_path) {
        FILE *f = fopen(csv_path, "a");
        if (f) {
            fseek(f, 0, SEEK_END);
            if (ftell(f) == 0)
                fprintf(f, "gpu,n,block,grid,kernel_mediana_ms,kernel_min_ms,kernel_max_ms,"
                           "con_copias_mediana_ms,ancho_banda_GB_s,verificacion\n");
            fprintf(f, "\"%s\",%d,%d,%d,%.4f,%.4f,%.4f,%.3f,%.1f,%s\n", prop.name, n, block, grid,
                    k_med, k_min, k_max, t_med, bw, ok ? "OK" : "FALLA");
            fclose(f);
        }
    }

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    free(h_a);
    free(h_b);
    free(h_c);
    free(h_ref);
    return ok ? 0 : 2;
}

In [ ]:
%%writefile index_probe.cu
/*
 * index_probe.cu — ¿Qué índice global le toca a cada hilo? (fase Investigar de PRIMM)
 *
 * Uso: ./index_probe [n] [block]      (por defecto n = 10, block = 4 -> 3 bloques, 12 hilos, 2 ociosos)
 * Compilar y correr (GPU): nvcc -O2 -arch=sm_75 index_probe.cu -o index_probe && ./index_probe 10 4
 *
 * Hace dos cosas:
 *  1. Cada hilo imprime con printf desde el kernel. El ORDEN de esas líneas no está garantizado:
 *     los bloques (y los warps dentro de ellos) corren en el orden que decida la GPU.
 *  2. Cada hilo con i < n anota en owner_block[i] y owner_thread[i] quién lo atendió; el anfitrión
 *     imprime esa tabla ordenada. Sirve para comprobar que cada índice 0..n-1 tiene exactamente un dueño.
 */
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                           \
    do {                                                                           \
        cudaError_t err__ = (call);                                                \
        if (err__ != cudaSuccess) {                                                \
            fprintf(stderr, "Error de CUDA en %s:%d: %s\n", __FILE__, __LINE__,   \
                    cudaGetErrorString(err__));                                    \
            exit(EXIT_FAILURE);                                                    \
        }                                                                          \
    } while (0)

__global__ void print_index(int n, int *owner_block, int *owner_thread)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        printf("bloque %d, hilo %d -> i = %d\n", blockIdx.x, threadIdx.x, i);
        owner_block[i] = blockIdx.x;
        owner_thread[i] = threadIdx.x;
    } else {
        printf("bloque %d, hilo %d -> i = %d  (i >= n: este hilo NO trabaja)\n", blockIdx.x, threadIdx.x, i);
    }
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : 10;
    int block = (argc > 2) ? atoi(argv[2]) : 4;
    if (n <= 0 || block <= 0 || block > 1024 || n > 4096) {
        fprintf(stderr, "uso: %s [n 1..4096] [block 1..1024]\n", argv[0]);
        return 1;
    }
    int grid = (n + block - 1) / block;
    printf("n=%d block=%d grid=%d  hilos_lanzados=%d  ociosos=%d\n", n, block, grid, grid * block,
           grid * block - n);

    int *d_owner_block = NULL, *d_owner_thread = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_owner_block, (size_t)n * sizeof(int)));
    CUDA_CHECK(cudaMalloc((void **)&d_owner_thread, (size_t)n * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_owner_block, -1, (size_t)n * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_owner_thread, -1, (size_t)n * sizeof(int)));

    printf("--- printf desde el kernel (orden NO garantizado) ---\n");
    print_index<<<grid, block>>>(n, d_owner_block, d_owner_thread);
    CUDA_CHECK(cudaGetLastError());      /* error de lanzamiento */
    CUDA_CHECK(cudaDeviceSynchronize()); /* espera al kernel y vacía el buffer de printf */

    int *owner_block = (int *)malloc((size_t)n * sizeof(int));
    int *owner_thread = (int *)malloc((size_t)n * sizeof(int));
    CUDA_CHECK(cudaMemcpy(owner_block, d_owner_block, (size_t)n * sizeof(int), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(owner_thread, d_owner_thread, (size_t)n * sizeof(int), cudaMemcpyDeviceToHost));

    printf("--- tabla ordenada por índice (la imprime el anfitrión) ---\n");
    int missing = 0;
    for (int i = 0; i < n; i++) {
        if (owner_block[i] < 0)
            missing++;
        printf("i = %d <- bloque %d, hilo %d\n", i, owner_block[i], owner_thread[i]);
    }
    printf("indices_sin_dueno=%d\n", missing);

    free(owner_block);
    free(owner_thread);
    CUDA_CHECK(cudaFree(d_owner_block));
    CUDA_CHECK(cudaFree(d_owner_thread));
    return 0;
}

In [ ]:
%%writefile vector_add_cpu.c
/*
 * vector_add_cpu.c — Referencia secuencial de la suma de vectores c = a + b.
 *
 * Uso: ./vector_add_cpu [n]      (por defecto n = 16 777 216 = 2^24)
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra vector_add_cpu.c -o vector_add_cpu
 *
 * Imprime el mismo checksum que vector_add.cu y vector_add_numba.py para el mismo n:
 * así se comparan las tres versiones aunque corran en máquinas distintas.
 * Medición: calentamiento + 5 repeticiones, mediana (protocolo del curso).
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define REPS 5

static double now_s(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

/* El ciclo que la GPU reparte entre miles de hilos: aquí lo hace un solo núcleo. */
static void vector_add_cpu(const float *a, const float *b, float *c, int n)
{
    for (int i = 0; i < n; i++)
        c[i] = a[i] + b[i];
}

static int cmp_double(const void *x, const void *y)
{
    double dx = *(const double *)x, dy = *(const double *)y;
    return (dx > dy) - (dx < dy);
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : (1 << 24);
    if (n <= 0) {
        fprintf(stderr, "uso: %s [n>0]\n", argv[0]);
        return 1;
    }
    size_t bytes = (size_t)n * sizeof(float);
    float *a = (float *)malloc(bytes);
    float *b = (float *)malloc(bytes);
    float *c = (float *)malloc(bytes);
    if (!a || !b || !c) {
        fprintf(stderr, "Sin memoria para n=%d\n", n);
        return 1;
    }
    /* Misma inicialización que las versiones CUDA y Numba. */
    for (int i = 0; i < n; i++) {
        a[i] = (float)(i % 7);
        b[i] = (float)(i % 13);
    }
    memset(c, 0, bytes);

    vector_add_cpu(a, b, c, n); /* calentamiento (no se cuenta) */

    double t[REPS];
    for (int r = 0; r < REPS; r++) {
        double t0 = now_s();
        vector_add_cpu(a, b, c, n);
        t[r] = now_s() - t0;
    }
    qsort(t, REPS, sizeof(double), cmp_double);
    double t_med = t[REPS / 2];

    /* Autoverificación contra la fórmula cerrada y checksum reproducible. */
    double checksum = 0.0, max_abs_err = 0.0;
    for (int i = 0; i < n; i++) {
        double expected = (double)(i % 7) + (double)(i % 13);
        double diff = fabs((double)c[i] - expected);
        if (diff > max_abs_err)
            max_abs_err = diff;
        checksum += c[i];
    }
    int ok = (max_abs_err <= 1e-5);
    double bw = (3.0 * (double)bytes / 1e9) / (t_med > 0 ? t_med : 1e-12);

    printf("vector_add_cpu: n=%d reps=%d mediana_ms=%.4f (min %.4f, max %.4f)  ancho_banda_efectivo_GB_s=%.1f\n",
           n, REPS, t_med * 1e3, t[0] * 1e3, t[REPS - 1] * 1e3, bw);
    printf("checksum=%.1f max_abs_err=%.3g verificacion=%s\n", checksum, max_abs_err, ok ? "OK" : "FALLA");

    free(a);
    free(b);
    free(c);
    return ok ? 0 : 2;
}

In [ ]:
%%writefile vector_add_numba.py
#!/usr/bin/env python3
"""vector_add_numba.py — Espejo en Numba CUDA del kernel de vector_add.cu.

Uso:
    python3 vector_add_numba.py [n] [block] [grid] [--sin-guardia]
Sin GPU (simulador en CPU, solo correctitud; n pequeño):
    NUMBA_ENABLE_CUDASIM=1 python3 vector_add_numba.py 1000 256

Mismo índice global, misma guardia, misma inicialización y mismo checksum que la versión en C.
En GPU real el tiempo incluye una compilación JIT la primera vez; por eso hay calentamiento.
"""
import os
import sys
import time

import numpy as np
from numba import cuda

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 5


@cuda.jit
def vector_add(a, b, c, n):
    # Índice global igual que en CUDA C. (cuda.grid(1) es la abreviatura equivalente.)
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:  # guardia
        c[i] = a[i] + b[i]


@cuda.jit
def vector_add_sin_guardia(a, b, c, n):
    # Variante deliberadamente incorrecta: los hilos sobrantes escriben fuera del arreglo.
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    c[i] = a[i] + b[i]


def median(values):
    return sorted(values)[len(values) // 2]


def main():
    flags = [x for x in sys.argv[1:] if x.startswith("--")]
    args = [x for x in sys.argv[1:] if not x.startswith("--")]
    sin_guardia = "--sin-guardia" in flags

    # Bajo CUDASIM cada hilo se simula en Python: con n grande tardaría minutos.
    default_n = 2048 if CUDASIM else 1 << 24
    n = int(args[0]) if len(args) > 0 else default_n
    block = int(args[1]) if len(args) > 1 else 256
    grid = int(args[2]) if len(args) > 2 else 0
    if n <= 0 or block <= 0 or block > 1024:
        print("uso: vector_add_numba.py [n>0] [block 1..1024] [grid; 0=auto] [--sin-guardia]")
        sys.exit(1)
    if grid <= 0:
        grid = (n + block - 1) // block  # división entera redondeada hacia arriba

    modo = "CUDASIM" if CUDASIM else "GPU"
    if CUDASIM and n > 20000:
        print("aviso: CUDASIM con n > 20000 puede tardar varios minutos", file=sys.stderr)

    # Anfitrión: misma inicialización que vector_add.cu / vector_add_cpu.c
    a = (np.arange(n) % 7).astype(np.float32)
    b = (np.arange(n) % 13).astype(np.float32)
    c = np.zeros(n, dtype=np.float32)

    # Dispositivo: equivalentes de cudaMalloc + cudaMemcpy H2D
    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_c = cuda.to_device(c)

    kernel = vector_add_sin_guardia if sin_guardia else vector_add

    # Calentamiento (en GPU real incluye la compilación JIT del kernel)
    kernel[grid, block](d_a, d_b, d_c, n)
    cuda.synchronize()

    times_ms = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        kernel[grid, block](d_a, d_b, d_c, n)
        cuda.synchronize()  # el lanzamiento es asíncrono: sin esto se mide casi nada
        times_ms.append((time.perf_counter() - t0) * 1e3)

    d_c.copy_to_host(c)  # equivalente de cudaMemcpy D2H

    # Verificación contra la CPU (NumPy): "sin error" no significa "correcto"
    ref = a + b
    max_abs_err = float(np.max(np.abs(c - ref)))
    checksum = float(np.sum(c, dtype=np.float64))
    ok = max_abs_err <= 1e-5

    print(f"vector_add_numba: n={n} block={block} grid={grid} hilos_lanzados={grid * block} modo={modo}")
    print(f"kernel_mediana_ms={median(times_ms):.4f} (min {min(times_ms):.4f}, max {max(times_ms):.4f})")
    print(f"checksum={checksum:.1f} max_abs_err={max_abs_err:.3g} verificacion={'OK' if ok else 'FALLA'}")
    sys.exit(0 if ok else 2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile index_probe_numba.py
#!/usr/bin/env python3
"""index_probe_numba.py — Espejo en Numba de index_probe.cu: ¿qué índice global le toca a cada hilo?

Uso:  NUMBA_ENABLE_CUDASIM=1 python3 index_probe_numba.py [n] [block]     (por defecto 10 y 4)
En GPU real también funciona.

1. Cada hilo imprime desde el kernel (orden NO garantizado; en CUDASIM las líneas incluso se
   pueden entremezclar porque el simulador usa hilos de Python).
2. Cada hilo con i < n anota quién lo atendió en owner_block[i], owner_thread[i]; el anfitrión
   imprime esa tabla ordenada para comprobar que cada índice tiene exactamente un dueño.
"""
import sys

import numpy as np
from numba import cuda


@cuda.jit
def print_index(n, owner_block, owner_thread):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:
        print("bloque", cuda.blockIdx.x, ", hilo", cuda.threadIdx.x, "-> i =", i)
        owner_block[i] = cuda.blockIdx.x
        owner_thread[i] = cuda.threadIdx.x
    else:
        print("bloque", cuda.blockIdx.x, ", hilo", cuda.threadIdx.x, "-> i =", i, "(i >= n: este hilo NO trabaja)")


def main():
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 10
    block = int(sys.argv[2]) if len(sys.argv) > 2 else 4
    if n <= 0 or block <= 0 or block > 1024 or n > 4096:
        print("uso: index_probe_numba.py [n 1..4096] [block 1..1024]")
        sys.exit(1)
    grid = (n + block - 1) // block
    print(f"n={n} block={block} grid={grid}  hilos_lanzados={grid * block}  ociosos={grid * block - n}")

    owner_block = np.full(n, -1, dtype=np.int32)
    owner_thread = np.full(n, -1, dtype=np.int32)
    d_owner_block = cuda.to_device(owner_block)
    d_owner_thread = cuda.to_device(owner_thread)

    print("--- print desde el kernel (orden NO garantizado) ---", flush=True)
    print_index[grid, block](n, d_owner_block, d_owner_thread)
    cuda.synchronize()  # espera al kernel; en GPU real además vacía el buffer de print
    sys.stdout.flush()

    d_owner_block.copy_to_host(owner_block)
    d_owner_thread.copy_to_host(owner_thread)

    print("--- tabla ordenada por índice (la imprime el anfitrión) ---")
    for i in range(n):
        print(f"i = {i} <- bloque {owner_block[i]}, hilo {owner_thread[i]}")
    print(f"indices_sin_dueno={int(np.sum(owner_block < 0))}")


if __name__ == "__main__":
    main()

## 2. Compilar

`nvcc` compila el código de anfitrión (con el compilador de C++ del sistema) y el de dispositivo (a código de la GPU). En una T4 la línea equivale a `nvcc -O2 -arch=sm_75 vector_add.cu -o vector_add`.

In [ ]:
# En una T4 (Colab) esto equivale a:  !nvcc -O2 -arch=sm_75 vector_add.cu -o vector_add
!nvcc -O2 -arch={ARCH} vector_add.cu -o vector_add
!nvcc -O2 -arch={ARCH} index_probe.cu -o index_probe
!gcc -std=c11 -O2 -Wall -Wextra vector_add_cpu.c -o vector_add_cpu -lm
!ls -l vector_add index_probe vector_add_cpu

## 3. Ejecutar (fase *Run* de PRIMM)

Antes de cada celda anota tu predicción en el README. Primero la sonda de índices con `n=10`, `block=4`; después la suma con `n=1000`; después el tamaño por defecto (2^24 = 16 777 216 elementos).

In [ ]:
!./index_probe 10 4

In [ ]:
!./vector_add 1000 256

In [ ]:
!./vector_add_cpu
!./vector_add

El espejo en Numba hace lo mismo desde Python. En GPU real la primera llamada compila el kernel (JIT); por eso el programa calienta antes de medir. Sin GPU, en tu laptop: `NUMBA_ENABLE_CUDASIM=1 python3 vector_add_numba.py 1000 256`.

In [ ]:
!python3 vector_add_numba.py
!python3 index_probe_numba.py 10 4

## 4. Investigar

**a) Grid insuficiente.** Forzamos 3 bloques de 256 para `n=1000` (768 hilos). No hay error de CUDA. ¿Qué dice la verificación? ¿Qué elementos quedaron sin calcular?

**b) Sin guardia.** Compilamos con `-DSIN_GUARDIA`: 4 bloques de 256 = 1024 hilos para `n=1000`; 24 hilos escriben fuera del arreglo. Observa si CUDA reporta algo. Repite con `n=1000 block=1024 grid=4000` (mucho más allá del arreglo) y compara.

In [ ]:
!./vector_add 1000 256 3; echo "código de salida: $?"

In [ ]:
!nvcc -O2 -arch={ARCH} -DSIN_GUARDIA vector_add.cu -o vector_add_sin_guardia
!./vector_add_sin_guardia 1000 256; echo "código de salida: $?"
!./vector_add_sin_guardia 1000 1024 4000; echo "código de salida: $?"

## 5. Medir (protocolo del curso)

`vector_add` ya hace calentamiento + 5 repeticiones y reporta la mediana, el tiempo del kernel solo y el tiempo con copias anfitrión↔dispositivo. Aquí barremos el tamaño de bloque con `n = 2^24` y guardamos cada fila en `resultados.csv`. Anota el modelo de GPU y la fecha: el hardware que asigna Colab cambia.

In [ ]:
import os, subprocess
if os.path.exists("resultados.csv"):
    os.remove("resultados.csv")
N = 16777216
for block in [32, 64, 128, 256, 512, 1024]:
    out = subprocess.run(["./vector_add", str(N), str(block), "0", "resultados.csv"],
                         capture_output=True, text=True)
    print(out.stdout.strip().splitlines()[-2])  # línea de tiempos
    print(out.stdout.strip().splitlines()[-1])  # línea de verificación
    if out.returncode != 0:
        print("stderr:", out.stderr)

## 6. Guardar resultados

La celda final muestra la tabla, agrega la fecha y descarga el CSV (en Kaggle queda en `/kaggle/working/`). Súbelo a tu repositorio en `resultados/`. Para tu bitácora: ¿cuántas veces más tarda la versión *con copias* que el kernel solo? ¿Y comparado con `vector_add_cpu`? Eso es el tema de la sesión S04 (lunes 30 de noviembre).

In [ ]:
import datetime
import pandas as pd
df = pd.read_csv("resultados.csv")
df["fecha"] = datetime.date.today().isoformat()
print(df.to_string(index=False))
nombre = f"resultados_S01_vector_add_{datetime.date.today().isoformat()}.csv"
df.to_csv(nombre, index=False)
print("guardado:", nombre)
try:
    from google.colab import files  # solo existe en Colab
    files.download(nombre)
except ImportError:
    print("En Kaggle: descarga el archivo desde el panel Output (/kaggle/working/).")